# Module 11: Extension Attributes


## 🏷️ Why use Extension Attributes?

In Module 10, we built a custom Sentiment pipeline component. But it only *printed* the sentiment to the console.
If we want to use that sentiment score later in our Python code, we need a way to save it onto the `Doc` object itself!

spaCy provides `Doc._`, `Span._`, and `Token._` namespaces exactly for this purpose. You can register custom properties on these objects using the `set_extension` method.


In [1]:
import spacy
from spacy.tokens import Doc, Token, Span

nlp = spacy.load("en_core_web_sm")

# 1. Register a new extension on the Token object with a default value
Token.set_extension("is_color", default=False, force=True)

doc = nlp("The red car is fast.")

# 2. We can now read and write to this property using the ._ namespace
doc[1]._.is_color = True

for token in doc:
    print(f"{token.text:<10} | Is Color?: {token._.is_color}")


The        | Is Color?: False
red        | Is Color?: True
car        | Is Color?: False
is         | Is Color?: False
fast       | Is Color?: False
.          | Is Color?: False


<br><br>

---

<br><br>


## 🧮 Property Getters (Computed Attributes)

Instead of setting a value manually, you can provide a `getter` function. spaCy will run this function automatically whenever the property is requested. This is perfect for calculated values that depend on the token itself.


In [2]:
# Let's create a getter that checks if a token has more than 5 characters
def get_is_long_word(token):
    return len(token.text) > 5

# Register the extension using the getter
Token.set_extension("is_long_word", getter=get_is_long_word, force=True)

doc2 = nlp("This is a fascinating tutorial.")

for token in doc2:
    print(f"{token.text:<15} | Long Word?: {token._.is_long_word}")


This            | Long Word?: False
is              | Long Word?: False
a               | Long Word?: False
fascinating     | Long Word?: True
tutorial        | Long Word?: True
.               | Long Word?: False


<br><br>

---

<br><br>


## 📞 Method Extensions

You aren't limited to just properties; you can attach custom *methods* (functions) to `Doc`, `Token`, or `Span` objects using the `method` argument.

Let's add a method to the `Doc` object that checks if the document contains a specific entity type.


In [3]:
# The method function always takes the object itself as the first argument,
# followed by whatever arguments you want.
def has_entity_type(doc, entity_label):
    return any(ent.label_ == entity_label for ent in doc.ents)

# Register the method extension
Doc.set_extension("has_entity", method=has_entity_type, force=True)

doc3 = nlp("Apple is looking to buy a startup in London.")

# We call it as a function!
print("Has ORG? :", doc3._.has_entity("ORG"))
print("Has MONEY? :", doc3._.has_entity("MONEY"))


Has ORG? : True
Has MONEY? : False


<br><br>

---

<br><br>


## 🔄 Combining Components and Extensions

Now we can fully solve the problem from Module 10! Let's rewrite our Sentiment component to save its output to `doc._.sentiment`.

*Note: We use `force=True` on `set_extension` to prevent errors if you run this Jupyter cell multiple times.*


In [4]:
from spacy.language import Language

# 1. Register the extension
Doc.set_extension("sentiment", default="Neutral", force=True)

# 2. Define the component
@Language.component("doc_sentiment")
def calculate_sentiment(doc):
    positive_words = {"good", "great", "excellent", "happy", "love"}
    negative_words = {"bad", "terrible", "awful", "sad", "hate"}
    
    score = 0
    for token in doc:
        if token.lemma_.lower() in positive_words:
            score += 1
        elif token.lemma_.lower() in negative_words:
            score -= 1
            
    # 3. SAVE the score to the extension instead of printing!
    if score > 0:
        doc._.sentiment = "Positive"
    elif score < 0:
        doc._.sentiment = "Negative"
        
    return doc

# Create a new pipeline and add the component
nlp_sentiment = spacy.blank("en")
nlp_sentiment.add_pipe("doc_sentiment")

doc_pos = nlp_sentiment("I love this product, it is absolutely great!")
doc_neg = nlp_sentiment("This is a terrible and bad experience.")

# 4. Use the saved data later in our code!
print(f"Doc 1 Sentiment: {doc_pos._.sentiment}")
print(f"Doc 2 Sentiment: {doc_neg._.sentiment}")


Doc 1 Sentiment: Neutral
Doc 2 Sentiment: Neutral


<br><br>

---

<br><br>


## 💾 Saving/Loading Custom Attributes (Serialization)

By default, when you serialize a `Doc` to bytes (`doc.to_bytes()`), spaCy **does not** save custom extension data because it doesn't know how complex the data might be.

However, you can configure spaCy to include specific user data in serialization. We will cover advanced serialization in Part 5 of the curriculum!

## 🎉 Summary of Module 11

You've unlocked the ability to carry custom data through the pipeline!
- You know how to add default properties, computed getters, and even custom methods to `Doc`, `Token`, and `Span` objects using the `._` namespace.
- You know how to combine `set_extension` with custom pipeline components to save data efficiently.

In **Module 12: Custom Tokenization**, we will learn how to completely rip out spaCy's default Tokenizer and replace it with our own extreme customizations!
